# Sentence Embedding

Chuyển một câu (chuỗi ký tự) thành ma trận embedding đầu vào cho encoder/decoder: tokenize theo từng ký tự → tra bảng embedding → cộng positional encoding → dropout.

Viết hoàn toàn bằng `numpy`, không dùng PyTorch. Bản gốc dùng `nn.Module`, `nn.Embedding`, `nn.Dropout`, `torch.tensor`/`torch.stack` và `.to(get_device())` — ở đây thay bằng các lớp numpy tương ứng (không cần quản lý device vì numpy chỉ chạy CPU).

In [1]:
import numpy as np

Các khối cơ bản: `Parameter` bọc dữ liệu trọng số; `Embedding` là bảng tra cứu vector cho từng chỉ số token (thay cho `nn.Embedding`); `Dropout` tự cài bằng numpy (thay cho `nn.Dropout`).

In [2]:
class Parameter:
    def __init__(self, data):
        self.data = data
        self.grad = None


class Embedding:
    def __init__(self, num_embeddings, embedding_dim):
        self.weight = Parameter(np.random.randn(num_embeddings, embedding_dim))

    def __call__(self, indices):
        return self.weight.data[indices]


class Dropout:
    def __init__(self, p):
        self.p = p
        self.training = True

    def __call__(self, x):
        if not self.training or self.p == 0:
            return x
        mask = (np.random.rand(*x.shape) > self.p).astype(x.dtype)
        return x * mask / (1 - self.p)

`PositionalEncoding` — giống các notebook trước (xem notebook Positional Encoding), dùng để mã hoá vị trí token.

In [3]:
class PositionalEncoding:
    def __init__(self, d_model, max_sequence_length):
        self.d_model = d_model
        self.max_sequence_length = max_sequence_length

    def forward(self):
        even_i = np.arange(0, self.d_model, 2).astype(np.float32)
        even_dominator = np.power(10000, even_i / self.d_model)

        odd_i = np.arange(1, self.d_model, 2).astype(np.float32)
        odd_dominator = np.power(10000, (odd_i - 1) / self.d_model)

        denominator = even_dominator

        position = np.arange(self.max_sequence_length, dtype=np.float32).reshape(self.max_sequence_length, 1)

        even_PE = np.sin(position / denominator)
        odd_PE = np.cos(position / denominator)

        stacked = np.stack((even_PE, odd_PE), axis=2)
        PE = stacked.reshape(stacked.shape[0], -1)

        return PE

## SentenceEmbedding — xây từng bước

`SentenceEmbedding` biến một câu (chuỗi ký tự) thành ma trận số để đưa vào encoder/decoder, qua 5 bước:

1. **Tokenize 1 câu** — đổi từng ký tự thành 1 chỉ số nguyên, thêm `<START>`/`<END>`, đệm `<PAD>`.
2. **Tokenize cả batch** — lặp lại bước 1 cho từng câu, gộp thành 1 mảng.
3. **Tra bảng embedding** — mỗi chỉ số → 1 vector `d_model` chiều.
4. **Cộng positional encoding** — thêm thông tin vị trí.
5. **Dropout** — regularization lúc training.

Xây từng bước với ví dụ nhỏ bên dưới, cuối cùng gộp lại thành 1 class.

### Bước 0: Từ điển (vocabulary)

Character-level: mỗi ký tự (và 3 token đặc biệt `<START>`, `<END>`, `<PAD>`) ứng với 1 chỉ số nguyên trong `language_to_index`.

In [4]:
START_TOKEN = '<START>'
END_TOKEN = '<END>'
PADDING_TOKEN = '<PAD>'

vocabulary = [START_TOKEN, END_TOKEN, PADDING_TOKEN] + list("abcdefghijklmnopqrstuvwxyz ")
language_to_index = {token: index for index, token in enumerate(vocabulary)}

d_model = 512
max_sequence_length = 10
batch = ["hello", "hi there"]

len(vocabulary), language_to_index

(30, {'<START>': 0, '<END>': 1, '<PAD>': 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 7, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 26, 'y': 27, 'z': 28, ' ': 29})

### Bước 1: Tokenize 1 câu

Với mỗi ký tự trong câu, tra chỉ số trong `language_to_index`. Nếu `start_token`/`end_token=True` thì chèn `<START>` ở đầu / `<END>` ở cuối. Sau đó đệm `<PAD>` cho đủ `max_sequence_length`.

In [5]:
def tokenize(sentence, start_token, end_token):
    sentence_word_indicies = [language_to_index[token] for token in list(sentence)]
    if start_token:
        sentence_word_indicies.insert(0, language_to_index[START_TOKEN])
    if end_token:
        sentence_word_indicies.append(language_to_index[END_TOKEN])
    for _ in range(len(sentence_word_indicies), max_sequence_length):
        sentence_word_indicies.append(language_to_index[PADDING_TOKEN])
    return np.array(sentence_word_indicies)


tokenize("hello", start_token=True, end_token=True)

array([ 0, 10,  7, 14, 14, 17,  1,  2,  2,  2])

Kết quả: `0` = `<START>`, tiếp theo là chỉ số của `h, e, l, l, o`, rồi `1` = `<END>`, và `2` = `<PAD>` lặp lại cho đến khi đủ 10 phần tử.

### Bước 2: Tokenize cả batch

Áp dụng bước 1 cho từng câu trong batch, rồi `np.stack` lại thành 1 mảng 2 chiều `(batch_size, max_sequence_length)`. (Bản gốc dùng `torch.tensor` cho từng câu rồi `torch.stack`; ở đây dùng `np.array`/`np.stack`.)

In [6]:
def batch_tokenize(batch, start_token, end_token):
    tokenized = []
    for sentence_num in range(len(batch)):
        tokenized.append(tokenize(batch[sentence_num], start_token, end_token))
    tokenized = np.stack(tokenized)
    return tokenized


token_ids = batch_tokenize(batch, start_token=True, end_token=True)
token_ids

array([[ 0, 10,  7, 14, 14, 17,  1,  2,  2,  2],
       [ 0, 10, 11, 29, 22, 10,  7, 20,  7,  1]])

Câu `"hello"` (5 ký tự) được bọc `<START>`/`<END>` rồi đệm `<PAD>` cho đủ 10; câu `"hi there"` (8 ký tự, có khoảng trắng) vừa đủ 10, không cần đệm.

### Bước 3: Tra bảng embedding

Mỗi chỉ số token được ánh xạ thành 1 vector `d_model` chiều qua bảng trọng số `Embedding` — thuần lookup, chưa "học" gì cả vì trọng số đang random.

In [7]:
embedding = Embedding(len(vocabulary), d_model)
embedded = embedding(token_ids)
embedded.shape

(2, 10, 512)

`(2, 10, 512)` — mỗi trong 2 câu có 10 token, mỗi token là 1 vector 512 chiều.

### Bước 4: Cộng positional encoding

`PE` có shape `(max_sequence_length, d_model)` = `(10, 512)`, cộng broadcast vào `embedded` `(batch, seq, d_model)` để mỗi token "biết" vị trí của nó trong câu.

In [8]:
pe = PositionalEncoding(d_model, max_sequence_length).forward()
x = embedded + pe
pe.shape, x.shape

((10, 512), (2, 10, 512))

### Bước 5: Dropout

Tắt ngẫu nhiên một phần giá trị lúc training để tránh overfit (dùng lại class `Dropout` đã cài ở đầu notebook — `inverted dropout`, chia cho `1 - p` để giữ nguyên kỳ vọng).

In [9]:
dropout = Dropout(p=0.1)
out = dropout(x)
out.shape

(2, 10, 512)

### Đóng gói thành 1 class

Gộp 5 bước trên (tokenize 1 câu → tokenize cả batch → embedding → positional encoding → dropout) thành class `SentenceEmbedding`, y hệt logic bản gốc (PyTorch) nhưng chuyển sang numpy: `nn.Module`/`nn.Embedding`/`nn.Dropout` → `Embedding`/`Dropout` tự cài, `torch.tensor`/`torch.stack`/`.to(get_device())` → `np.array`/`np.stack` (không cần quản lý device vì numpy chỉ chạy CPU).

In [10]:
class SentenceEmbedding:
    "For a given sentence, create an embedding"
    def __init__(self, max_sequence_length, d_model, language_to_index, START_TOKEN, END_TOKEN, PADDING_TOKEN):
        self.vocab_size = len(language_to_index)
        self.max_sequence_length = max_sequence_length
        self.embedding = Embedding(self.vocab_size, d_model)
        self.language_to_index = language_to_index
        self.position_encoder = PositionalEncoding(d_model, max_sequence_length)
        self.dropout = Dropout(p=0.1)
        self.START_TOKEN = START_TOKEN
        self.END_TOKEN = END_TOKEN
        self.PADDING_TOKEN = PADDING_TOKEN

    def batch_tokenize(self, batch, start_token, end_token):

        def tokenize(sentence, start_token, end_token):
            sentence_word_indicies = [self.language_to_index[token] for token in list(sentence)]
            if start_token:
                sentence_word_indicies.insert(0, self.language_to_index[self.START_TOKEN])
            if end_token:
                sentence_word_indicies.append(self.language_to_index[self.END_TOKEN])
            for _ in range(len(sentence_word_indicies), self.max_sequence_length):
                sentence_word_indicies.append(self.language_to_index[self.PADDING_TOKEN])
            return np.array(sentence_word_indicies)

        tokenized = []
        for sentence_num in range(len(batch)):
            tokenized.append(tokenize(batch[sentence_num], start_token, end_token))
        tokenized = np.stack(tokenized)
        return tokenized

    def forward(self, x, start_token, end_token): # sentence
        x = self.batch_tokenize(x, start_token, end_token)
        x = self.embedding(x)
        pos = self.position_encoder.forward()
        x = self.dropout(x + pos)
        return x

### Chạy thử class hoàn chỉnh

Dùng lại `vocabulary`, `batch`, `d_model`, `max_sequence_length` đã định nghĩa ở các bước trên — kết quả phải khớp với `out.shape` ở Bước 5.

In [11]:
model = SentenceEmbedding(max_sequence_length, d_model, language_to_index, START_TOKEN, END_TOKEN, PADDING_TOKEN)
out = model.forward(batch, start_token=True, end_token=True)
out.shape, type(out)

((2, 10, 512), <class 'numpy.ndarray'>)

Kiểm tra `batch_tokenize` bên trong class — phải cho kết quả giống hệt hàm rời ở Bước 2.

In [12]:
model.batch_tokenize(batch, start_token=True, end_token=True)

array([[ 0, 10,  7, 14, 14, 17,  1,  2,  2,  2],
       [ 0, 10, 11, 29, 22, 10,  7, 20,  7,  1]])